# 0. Load imports 

In [1]:
import pandas as pd
import numpy as np
import re
import gdown #needed to get data from google drive

import matplotlib.pyplot as plt
import seaborn as sns

## print multiple things from same cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

gdown.download(id="1kLQwTZiHZ2h-RdXfFmq8csGqXlPO4r9U", output="df_clean.csv", quiet=False)
df = pd.read_csv("df_clean.csv")

print(df.shape)
print(df.head())

Downloading...
From (original): https://drive.google.com/uc?id=1kLQwTZiHZ2h-RdXfFmq8csGqXlPO4r9U
From (redirected): https://drive.google.com/uc?id=1kLQwTZiHZ2h-RdXfFmq8csGqXlPO4r9U&confirm=t&uuid=ee497174-3de2-43d8-b134-79e9c6549ce5
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/df_clean.csv
100%|████████████████████████████████████████| 188M/188M [00:27<00:00, 6.74MB/s]


(378081, 30)
   undl_id ms_code              ms_name ms_vote        date session  \
0   507407     AFG          AFGHANISTAN       Y  2003-12-03      58   
1   507407     ALB              ALBANIA       Y  2003-12-03      58   
2   507407     DZA              ALGERIA       Y  2003-12-03      58   
3   507407     AND              ANDORRA       Y  2003-12-03      58   
4   507407     ATG  ANTIGUA AND BARBUDA       Y  2003-12-03      58   

    resolution                      draft committee_report     meeting  ...  \
0  A/RES/58/20  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68  ...   
1  A/RES/58/20  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68  ...   
2  A/RES/58/20  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68  ...   
3  A/RES/58/20  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68  ...   
4  A/RES/58/20  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68  ...   

   year ms_vote_score                       category  us_vote_score  \
0  2003       

/var/folders/cs/x74m_3cj4xv9_rl_f_d37ts40000gn/T/ipykernel_73454/3483895767.py:14: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("df_clean.csv")


# 1. Look at aid as a % of GDP in 2024 and 2025 to see which countries saw the largest fall to use for sentiment analysis

In [17]:
# get one row per country-year (ratio is same across all votes for that country-year)
aid_ratio = df[['ms_name', 'year', 'aid_gdp_ratio']].drop_duplicates()

# pivot to get 2024 and 2025 side by side
aid_2024 = aid_ratio[aid_ratio['year'] == 2024].set_index('ms_name')['aid_gdp_ratio']
aid_2025 = aid_ratio[aid_ratio['year'] == 2025].set_index('ms_name')['aid_gdp_ratio']

ratio_change = pd.DataFrame({'ratio_2024': aid_2024, 'ratio_2025': aid_2025}).dropna()
ratio_change['pct_point_change'] = ratio_change['ratio_2025'] - ratio_change['ratio_2024']

# biggest drops
print(ratio_change.sort_values('pct_point_change').head(40))

                                  ratio_2024  ratio_2025  pct_point_change
ms_name                                                                   
MARSHALL ISLANDS                  150.204147   97.852206        -52.351942
PALAU                              36.239114   26.055509        -10.183605
MICRONESIA (FEDERATED STATES OF)   77.143602   72.931610         -4.211993
SOMALIA                             8.043437    3.954794         -4.088642
SOUTH SUDAN                         7.013478    3.962021         -3.051457
LESOTHO                             4.938364    2.416217         -2.522147
UKRAINE                             5.326919    3.398709         -1.928211
CENTRAL AFRICAN REPUBLIC            4.281262    2.361252         -1.920010
YEMEN                               2.941885    1.219184         -1.722701
MOZAMBIQUE                          3.358599    1.953763         -1.404836
MALAWI                              3.558498    2.162313         -1.396186
ISRAEL                   

# 2. Bring in newspaper headline csvs for 3 countries which have a high amount of foreign assistance from the US

In [12]:
gdown.download(id="1B7gITHLFPruqIy7tqc2ZfK-KfHmnxwQd", output="jordan_headlines.csv", quiet=False)
df_jordan = pd.read_csv("jordan_headlines.csv")

print(df_jordan.shape)
print(df_jordan.head())

gdown.download(id="1NbqLWExyNG9R8yuU1y0KYwMr1VEuf5eZ", output="ethiopia_headlines.csv", quiet=False)
df_ethiopia = pd.read_csv("ethiopia_headlines.csv")

gdown.download(id="1OSjK-Mj5gmsm93gg3bQMrXjZLjSNulKi", output="drc_headlines.csv", quiet=False)
df_drc = pd.read_csv("drc_headlines.csv")



Downloading...
From: https://drive.google.com/uc?id=1B7gITHLFPruqIy7tqc2ZfK-KfHmnxwQd
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/jordan_headlines.csv
100%|████████████████████████████████████████| 176k/176k [00:00<00:00, 3.19MB/s]


'jordan_headlines.csv'

Downloading...
From: https://drive.google.com/uc?id=1NbqLWExyNG9R8yuU1y0KYwMr1VEuf5eZ
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/ethiopia_headlines.csv
100%|████████████████████████████████████████| 137k/137k [00:00<00:00, 3.49MB/s]


'ethiopia_headlines.csv'

Downloading...
From: https://drive.google.com/uc?id=1OSjK-Mj5gmsm93gg3bQMrXjZLjSNulKi
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/drc_headlines.csv
100%|██████████████████████████████████████| 17.7k/17.7k [00:00<00:00, 19.3MB/s]


'drc_headlines.csv'

(433, 8)
                                                  id  \
0  c574b353d9c503d606832371df11353ff6342069908924...   
1  e2fff23f73e3aaab6207a13c215d92ede12117cebc5b7c...   
2  f49a5081e427417d68e95bd1d37a410f515afae2df1918...   
3  c59985ff81f4b298208de44062360b7b457339c83f2c75...   
4  b4a824f42f9e0e640a03d2a0548f04221fd7d61906fc6f...   

                       indexed_date language      media_name       media_url  \
0  2026-04-21 20:23:21.773004+00:00       ar  sarahanews.net  sarahanews.net   
1  2026-03-29 07:26:28.976524+00:00       ar        jo24.net        jo24.net   
2  2026-01-19 12:35:05.798250+00:00       ar  sarahanews.net  sarahanews.net   
3  2026-01-06 13:32:52.383467+00:00       ar     royanews.tv     royanews.tv   
4  2025-12-23 20:23:02.718145+00:00       en   ammonnews.net   ammonnews.net   

  publish_date                                              title  \
0   2026-04-21  أسئلة مفتوحة برسم الاجابه حول “الإغلاق المالي”...   
1   2026-03-29  عامٌ على وقف برامج 